## Criação de Subclasses a partir da Classificação Binária

Após os experimentos iniciais com três classes, observou-se que a categoria intermediária apresentava forte sobreposição com as demais classes. Essa dificuldade ficou evidente nos resultados dos modelos multiclasse, nos quais a classe intermediária era frequentemente confundida tanto com a classe adequada quanto com a classe não adequada.

Diante disso, foi adotada uma nova estratégia: em vez de tentar separar diretamente três classes desde o início, o problema foi reformulado inicialmente como uma classificação binária, considerando apenas duas categorias principais:

* **Adequada**
* **Não adequada**

Essa abordagem teve como objetivo reduzir a ambiguidade do problema, permitindo que o modelo aprendesse primeiro a separar as amostras em dois grupos mais bem definidos. Após essa etapa, foi possível investigar com mais detalhes o comportamento interno de cada uma dessas classes.

Para isso, foram utilizadas as probabilidades preditas pelo modelo binário. A ideia central foi analisar o grau de confiança do modelo em relação a cada amostra. Por exemplo, uma amostra classificada como **Adequada** com probabilidade muito alta pode ser interpretada como uma amostra fortemente adequada. Já uma amostra classificada como **Adequada** com probabilidade mais próxima da fronteira de decisão pode indicar uma condição adequada, porém mais próxima de uma situação de alerta.

Com base nesse raciocínio, as amostras da classe **Adequada** foram divididas em dois subgrupos:

* **Melhores adequadas**: amostras classificadas como adequadas com maior confiança pelo modelo.
* **Piores adequadas**: amostras classificadas como adequadas, porém mais próximas da fronteira com a classe não adequada.

Da mesma forma, as amostras da classe **Não adequada** também foram divididas em dois subgrupos:

* **Melhores não adequadas**: amostras classificadas como não adequadas, mas mais próximas da fronteira com a classe adequada.
* **Piores não adequadas**: amostras classificadas como não adequadas com maior confiança pelo modelo.

Essa divisão permitiu representar a qualidade da água como um espectro contínuo, e não apenas como categorias rígidas. A estrutura esperada passou a ser:

```text
Melhores adequadas
        ↓
Piores adequadas
        ↓
Melhores não adequadas
        ↓
Piores não adequadas
```

O principal objetivo dessa etapa foi investigar se a antiga classe intermediária poderia ser melhor compreendida como uma região de transição entre amostras adequadas e não adequadas. Nesse contexto, as subclasses **Piores adequadas** e **Melhores não adequadas** passaram a representar os casos mais próximos da fronteira de decisão, ou seja, amostras que não pertencem claramente aos extremos de qualidade.

Portanto, a criação das subclasses não teve como finalidade apenas aumentar o número de categorias, mas sim compreender melhor a estrutura dos dados e verificar se a região intermediária poderia ser reconstruída de forma mais coerente a partir da fronteira aprendida pelo modelo binário.


In [ ]:
# IMPORT DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/Cópia de amostra_rotulada_binaria_2000_2008.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/water_quality_2000_2008.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset Parquet carregado com sucesso.
Shape do dataset: (59896, 23)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status,Year
0,Canada,FISW_32,Lake,2003-12-02,0.043792,2.13333,9.824,0.00200,7.7900,12.00000,...,Excellent,1,1,1,1,2.0,1,5,Adequada,2003
1,Canada,IEEA_10_32,Lake,2001-06-08,0.015920,0.55000,9.824,0.00400,7.7900,16.80000,...,Excellent,1,1,1,1,2.0,1,5,Adequada,2001
2,Canada,CHRW-1876,River,2000-01-12,0.064400,10.87500,11.250,0.03590,8.2833,12.76150,...,Good,1,1,0,1,1.0,1,4,Não adequada,2000
3,Canada,ES063ESPFAA0000714,River,2004-01-12,1.071725,1.24444,5.850,0.20425,7.1000,18.32500,...,Fair,1,1,1,0,3.7,1,4,Não adequada,2004
4,Canada,CZPLA_391,River,2003-01-12,0.039740,1.83333,11.050,0.06100,7.7500,8.66667,...,Good,1,1,1,0,2.0,1,4,Não adequada,2003


In [ ]:
X = df[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type",
    "Nitrogen (mg/l)"
]]

y = df["conama_status"]

In [ ]:
# DIVISÃO TREINO/TESTE
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (47916, 5)
Teste: (11980, 5)


In [ ]:
# PRÉ-PROCESSAMENTO
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
# sem balanceamento
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 LGBMClassifier(n_jobs=-1, random_state=42, verbose=-1))])

In [ ]:
# MÉTRICAS DE TREINO sem balanceamento
y_train_pred = model.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)


Train Accuracy:
0.8183279071708823
Train Precision:
0.8188043665823861
Train Recall:
0.8183279071708823
Train F1:
0.8185585168843551

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.87      0.87      0.87     32935
Não adequada       0.71      0.71      0.71     14981

    accuracy                           0.82     47916
   macro avg       0.79      0.79      0.79     47916
weighted avg       0.82      0.82      0.82     47916

Train Confusion Matrix:
[[28512  4423]
 [ 4282 10699]]


In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.8065108514190317

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.86      0.85      0.86      8234
Não adequada       0.69      0.70      0.69      3746

    accuracy                           0.81     11980
   macro avg       0.77      0.78      0.78     11980
weighted avg       0.81      0.81      0.81     11980


Confusion Matrix:
[[7036 1198]
 [1120 2626]]


In [ ]:
# Classes do modelo
model.classes_

array(['Adequada', 'Não adequada'], dtype=object)

In [ ]:
idx_adequada = list(model.classes_).index("Adequada")
idx_nao_adequada = list(model.classes_).index("Não adequada")

idx_adequada, idx_nao_adequada

(0, 1)

In [ ]:
probas = model.predict_proba(X)

df["prob_adequada"] = probas[:, idx_adequada]
df["prob_nao_adequada"] = probas[:, idx_nao_adequada]

In [ ]:
df["subgrupo_qualidade"] = None

In [ ]:
mask_adequada = df["conama_status"] == "Adequada"

corte_adequada = df.loc[mask_adequada, "prob_adequada"].median()

df.loc[
    mask_adequada & (df["prob_adequada"] >= corte_adequada),
    "subgrupo_qualidade"
] = "Melhores adequadas"

df.loc[
    mask_adequada & (df["prob_adequada"] < corte_adequada),
    "subgrupo_qualidade"
] = "Piores adequadas"

In [ ]:
mask_nao_adequada = df["conama_status"] == "Não adequada"

corte_nao_adequada = df.loc[mask_nao_adequada, "prob_nao_adequada"].median()

df.loc[
    mask_nao_adequada & (df["prob_nao_adequada"] >= corte_nao_adequada),
    "subgrupo_qualidade"
] = "Piores não adequadas"

df.loc[
    mask_nao_adequada & (df["prob_nao_adequada"] < corte_nao_adequada),
    "subgrupo_qualidade"
] = "Melhores não adequadas"

In [ ]:
df["subgrupo_qualidade"].value_counts()

,count
subgrupo_qualidade,
Melhores adequadas,20585
Piores adequadas,20584
Piores não adequadas,9365
Melhores não adequadas,9362


In [ ]:
pd.crosstab(df["conama_status"], df["subgrupo_qualidade"])

subgrupo_qualidade,Melhores adequadas,Melhores não adequadas,Piores adequadas,Piores não adequadas
conama_status,,,,
Adequada,20585,0,20584,0
Não adequada,0,9362,0,9365


In [ ]:
df.to_parquet(
    "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_binaria_com_subgrupos.parquet",
    index=False
)

In [ ]:
df.columns

Index(['Country', 'Area', 'Waterbody Type', 'Date', 'Ammonia (mg/l)',
       'Biochemical Oxygen Demand (mg/l)', 'Dissolved Oxygen (mg/l)',
       'Orthophosphate (mg/l)', 'pH (ph units)', 'Temperature (cel)',
       'Nitrogen (mg/l)', 'Nitrate (mg/l)', 'CCME_Values', 'CCME_WQI', 'ph_ok',
       'od_ok', 'dbo_ok', 'nitrate_ok', 'ammonia_limit', 'ammonia_ok',
       'environmental_score', 'conama_status', 'Year', 'prob_adequada',
       'prob_nao_adequada', 'subgrupo_qualidade'],
      dtype='object')

In [ ]:
variaveis_rotulo = [
    "Ammonia (mg/l)",
    "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)",
    "pH (ph units)",
    "Nitrate (mg/l)"
]

df.groupby("subgrupo_qualidade")[variaveis_rotulo].mean().round(2)

,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),pH (ph units),Nitrate (mg/l)
subgrupo_qualidade,,,,,
Melhores adequadas,0.10,1.76,10.45,7.79,3.67
Melhores não adequadas,4.44,11.91,9.57,7.59,6.79
Piores adequadas,0.33,2.35,10.10,7.70,3.90
Piores não adequadas,5.60,15.86,10.18,7.63,11.03


In [ ]:
variaveis_modelo = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Nitrogen (mg/l)"

]

df.groupby("subgrupo_qualidade")[variaveis_modelo].mean().round(2)

,Temperature (cel),Orthophosphate (mg/l),Nitrogen (mg/l)
subgrupo_qualidade,,,
Melhores adequadas,10.78,0.11,3.74
Melhores não adequadas,13.09,1.47,7.89
Piores adequadas,12.94,1.29,6.96
Piores não adequadas,11.73,4.14,14.74


In [ ]:
import plotly.express as px

fig = px.histogram(
    df[df["conama_status"] == "Adequada"],
    x="prob_adequada",
    nbins=50,
    title="Distribuição das probabilidades da classe Adequada"
)

fig.add_vline(
    x=0.907923,
    line_dash="dash",
    annotation_text="Mediana"
)

fig.show()

In [ ]:
fig = px.histogram(
    df[df["conama_status"] == "Não adequada"],
    x="prob_nao_adequada",
    nbins=50,
    title="Distribuição das probabilidades da classe Não adequada"
)

fig.add_vline(
    x=0.669155,
    line_dash="dash",
    annotation_text="Mediana"
)

fig.show()

## Interpretação dos Cortes por Probabilidade e dos Histogramas

Após a geração das probabilidades pelo modelo binário, foi necessário definir um critério para separar internamente as amostras das classes **Adequada** e **Não adequada**. Para isso, foi utilizada a mediana das probabilidades dentro de cada classe.

No caso da classe **Adequada**, o modelo apresentou uma distribuição de probabilidades bastante concentrada em valores altos. A mediana da probabilidade de adequação foi aproximadamente **0,9079**, ou seja, cerca de **90,79%**. Isso significa que metade das amostras adequadas recebeu probabilidade igual ou superior a esse valor, enquanto a outra metade ficou abaixo dele.

A partir desse corte, as amostras foram divididas da seguinte forma:

```text
prob_adequada >= 0,9079 → Melhores adequadas
prob_adequada <  0,9079 → Piores adequadas
```

O histograma da classe **Adequada** mostra que a maior parte das amostras está concentrada próxima de probabilidades elevadas, especialmente acima de 0,85. Isso indica que o modelo binário teve maior segurança ao reconhecer essa classe. Esse comportamento também pode ser explicado pela maior quantidade de amostras adequadas no conjunto de dados, o que permitiu ao modelo aprender melhor os padrões associados a essa categoria.

Já na classe **Não adequada**, a distribuição das probabilidades foi mais espalhada. A mediana da probabilidade de não adequação foi aproximadamente **0,6691**, ou seja, cerca de **66,91%**. Dessa forma, o corte ficou menos elevado do que na classe adequada, indicando que o modelo tinha menor confiança média ao reconhecer as amostras não adequadas.

A divisão foi feita assim:

```text
prob_nao_adequada >= 0,6691 → Piores não adequadas
prob_nao_adequada <  0,6691 → Melhores não adequadas
```

O histograma da classe **Não adequada** confirma essa maior dispersão. Existem amostras com baixa, média e alta probabilidade de não adequação, o que mostra que essa classe é mais heterogênea. Isso está coerente com a forma como o rótulo binário foi criado, pois a classe **Não adequada** reuniu diferentes níveis de qualidade, incluindo casos mais próximos da fronteira com a classe adequada e casos mais críticos.

É importante destacar que os termos **Melhores** e **Piores** não significam, inicialmente, uma avaliação ambiental absoluta feita diretamente pelo modelo. Eles representam o grau de confiança do modelo dentro de cada classe. No entanto, ao comparar as médias das variáveis ambientais, observou-se que essa confiança também apresentou coerência com os indicadores de qualidade da água. As **Melhores adequadas** apresentaram melhores condições ambientais, enquanto as **Piores não adequadas** apresentaram os piores indicadores.

Outro ponto importante é que as quantidades entre os subgrupos ficaram praticamente iguais porque a mediana divide cada classe principal em duas partes. Por isso, a classe **Adequada** foi dividida em aproximadamente metade como **Melhores adequadas** e metade como **Piores adequadas**. O mesmo ocorreu com a classe **Não adequada**, dividida entre **Melhores não adequadas** e **Piores não adequadas**.

Essa etapa foi essencial para compreender melhor a estrutura interna das classes. A análise mostrou que a qualidade da água não se organiza apenas como uma divisão rígida entre adequada e não adequada, mas como uma gradação. As amostras de maior confiança representam os extremos da qualidade, enquanto as amostras de menor confiança ficam mais próximas da região de transição.

Assim, a região intermediária pode ser compreendida principalmente a partir da combinação entre:

```text
Piores adequadas
+
Melhores não adequadas
```

Esses dois grupos representam amostras próximas da fronteira entre condições adequadas e não adequadas. Essa interpretação serviu como base para a reconstrução posterior de um novo rótulo de três classes:

```text
Melhores adequadas        → Adequada
Piores adequadas          → Atenção
Melhores não adequadas    → Atenção
Piores não adequadas      → Não adequada
```

Com isso, a classe **Atenção** deixou de ser definida apenas por um corte fixo no score e passou a ser construída com base na fronteira aprendida pelo modelo. Essa abordagem tornou a classificação mais coerente com o comportamento real dos dados e ajudou a reduzir a sobreposição observada nos experimentos multiclasse anteriores.


### Análise dos Subgrupos Gerados a partir da Classificação Binária

Com o objetivo de compreender melhor o comportamento das amostras próximas à fronteira de decisão do modelo, foi realizada uma subdivisão das classes binárias "Adequada" e "Não adequada" utilizando as probabilidades preditas pelo modelo LightGBM. Dessa forma, cada classe foi dividida em dois subgrupos: "Melhores adequadas", "Piores adequadas", "Melhores não adequadas" e "Piores não adequadas".

A análise das variáveis ambientais utilizadas na construção do rótulo (amônia, demanda bioquímica de oxigênio - DBO, oxigênio dissolvido, nitrato e pH) revelou um comportamento consistente e coerente com os níveis de qualidade da água. Observou-se que as amostras classificadas como "Melhores adequadas" apresentaram os menores valores de amônia e DBO, além dos maiores valores de oxigênio dissolvido, indicando condições ambientais mais favoráveis. Por outro lado, as "Piores adequadas" apresentaram uma leve deterioração desses indicadores, embora ainda permanecessem dentro da categoria considerada adequada.

Comportamento semelhante foi observado entre os subgrupos da classe "Não adequada". As amostras classificadas como "Piores não adequadas" apresentaram os maiores níveis de amônia, DBO e nitrato, caracterizando condições ambientais significativamente mais degradadas quando comparadas às "Melhores não adequadas". Esses resultados demonstram que o modelo foi capaz de identificar diferentes níveis de qualidade mesmo dentro de uma mesma classe binária.

Outro aspecto relevante foi a análise das variáveis utilizadas como preditoras do modelo (temperatura, ortofosfato e nitrogênio). Observou-se que os grupos "Piores adequadas" e "Melhores não adequadas" apresentaram valores bastante próximos para essas variáveis, sugerindo a existência de uma região de transição entre as duas classes. Essa proximidade ajuda a explicar a dificuldade encontrada pelos modelos no cenário multiclasse, especialmente quando a categoria intermediária era considerada como uma classe independente.

Os resultados obtidos reforçam a hipótese levantada durante os experimentos anteriores de que a principal dificuldade do problema não estava associada apenas ao desbalanceamento das classes, mas também à sobreposição de características entre amostras próximas da fronteira de decisão. A adoção da abordagem binária permitiu reduzir essa ambiguidade, produzindo uma separação mais consistente entre as categorias e contribuindo para a melhoria do desempenho dos modelos de classificação.


In [ ]:
df_adequadas = df[
    df["subgrupo_qualidade"].isin([
        "Melhores adequadas",
        "Piores adequadas"
    ])
].copy()

X = df_adequadas[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type",
    "Nitrogen (mg/l)"
]]

y = df_adequadas["subgrupo_qualidade"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

In [ ]:
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

model_adequadas = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [ ]:
model_adequadas.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 LGBMClassifier(n_jobs=-1, random_state=42, verbose=-1))])

In [ ]:
y_train_pred = model_adequadas.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)


Train Accuracy:
0.9787763777136784
Train Precision:
0.9788198943269649
Train Recall:
0.9787763777136784
Train F1:
0.9787758923459392

Classification Report:
                    precision    recall  f1-score   support

Melhores adequadas       0.97      0.98      0.98     16468
  Piores adequadas       0.98      0.97      0.98     16467

          accuracy                           0.98     32935
         macro avg       0.98      0.98      0.98     32935
      weighted avg       0.98      0.98      0.98     32935

Train Confusion Matrix:
[[16197   271]
 [  428 16039]]


In [ ]:
y_pred = model_adequadas.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.9720670391061452

Classification Report:
                    precision    recall  f1-score   support

Melhores adequadas       0.97      0.98      0.97      4117
  Piores adequadas       0.98      0.97      0.97      4117

          accuracy                           0.97      8234
         macro avg       0.97      0.97      0.97      8234
      weighted avg       0.97      0.97      0.97      8234


Confusion Matrix:
[[4027   90]
 [ 140 3977]]


### Interpretação dos Resultados da Subdivisão das Classes Binárias

Com o objetivo de investigar a existência de diferentes níveis de qualidade dentro das classes binárias, as amostras classificadas como "Adequada" foram subdivididas em "Melhores adequadas" e "Piores adequadas" a partir das probabilidades estimadas pelo modelo. Em seguida, um novo modelo LightGBM foi treinado para distinguir esses dois grupos.

Os resultados obtidos foram expressivos, alcançando acurácia de 97,88% no conjunto de treinamento e 97,21% no conjunto de teste. Além disso, os valores de precisão, recall e F1-score permaneceram elevados e equilibrados entre as duas subclasses, indicando excelente capacidade de generalização e ausência de sobreajuste significativo.

Esses resultados demonstram que as subclasses identificadas não representam divisões aleatórias dos dados. Pelo contrário, os grupos apresentam padrões distintos e consistentes, capazes de serem reconhecidos pelo modelo com elevado grau de confiança. Isso sugere que existe uma gradação interna de qualidade mesmo entre amostras classificadas como adequadas.

A análise complementa os resultados obtidos anteriormente nos experimentos binários e multiclasse. Enquanto os modelos de três classes apresentavam dificuldades para separar adequadamente a classe intermediária, a subdivisão baseada nas probabilidades revelou que as amostras se organizam de forma contínua ao longo de um espectro de qualidade ambiental. Dessa forma, os resultados reforçam a hipótese de que a principal dificuldade observada nos experimentos multiclasse estava associada à sobreposição entre regiões próximas da fronteira de decisão, e não necessariamente à ausência de padrões discriminativos nos dados.

Em termos práticos, o experimento evidencia que a classe "Adequada" não é homogênea, possuindo diferentes níveis internos de qualidade que podem ser identificados com elevada precisão pelos modelos de aprendizado de máquina.


In [ ]:
df_nao_adequadas = df[
    df["subgrupo_qualidade"].isin([
        "Melhores não adequadas",
        "Piores não adequadas"
    ])
].copy()

X = df_nao_adequadas[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type",
    "Nitrogen (mg/l)"
]]

y = df_nao_adequadas["subgrupo_qualidade"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

In [ ]:
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

model_nao_adequadas = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [ ]:
model_nao_adequadas.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 LGBMClassifier(n_jobs=-1, random_state=42, verbose=-1))])

In [ ]:
y_train_pred = model_nao_adequadas.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)

Train Accuracy:
0.9859154929577465
Train Precision:
0.9859668172519728
Train Recall:
0.9859154929577465
Train F1:
0.9859151063654994

Classification Report:
                        precision    recall  f1-score   support

Melhores não adequadas       0.99      0.98      0.99      7489
  Piores não adequadas       0.98      0.99      0.99      7492

              accuracy                           0.99     14981
             macro avg       0.99      0.99      0.99     14981
          weighted avg       0.99      0.99      0.99     14981

Train Confusion Matrix:
[[7345  144]
 [  67 7425]]


In [ ]:
y_pred = model_nao_adequadas.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.9682327816337427

Classification Report:
                        precision    recall  f1-score   support

Melhores não adequadas       0.97      0.96      0.97      1873
  Piores não adequadas       0.96      0.97      0.97      1873

              accuracy                           0.97      3746
             macro avg       0.97      0.97      0.97      3746
          weighted avg       0.97      0.97      0.97      3746


Confusion Matrix:
[[1801   72]
 [  47 1826]]


## Interpretação Final dos Experimentos com Classes Binárias e Subgrupos

Após os experimentos iniciais com três classes — **Adequada**, **Boa** e **Não adequada** — observou-se que os modelos apresentavam dificuldade para separar corretamente a classe intermediária. Mesmo com diferentes estratégias de balanceamento, como pesos manuais, SMOTE e undersampling, a performance continuava instável, principalmente por causa da baixa precisão ou do baixo recall em algumas classes.

Esses resultados indicaram que o principal problema não estava apenas no desbalanceamento dos dados, mas sim na **sobreposição entre as classes**, especialmente na região intermediária entre amostras adequadas e não adequadas.

Diante disso, foi adotada uma nova abordagem baseada em classificação binária, considerando apenas duas classes principais:

* **Adequada**
* **Não adequada**

Essa mudança melhorou significativamente a performance dos modelos, pois reduziu a ambiguidade existente entre as três classes originais. Em vez de tentar obrigar o modelo a separar diretamente uma classe intermediária pouco definida, o problema passou a ser tratado inicialmente como uma decisão mais objetiva: a amostra apresenta ou não condições adequadas de qualidade da água.

A partir do modelo binário, foi realizada uma nova análise utilizando as probabilidades preditas pelo próprio modelo. As amostras classificadas como **Adequadas** foram divididas em:

* **Melhores adequadas**
* **Piores adequadas**

E as amostras classificadas como **Não adequadas** foram divididas em:

* **Melhores não adequadas**
* **Piores não adequadas**

Essa divisão permitiu investigar a existência de níveis internos dentro das duas classes principais. Os resultados mostraram que essa subdivisão não foi aleatória: os novos modelos conseguiram distinguir esses subgrupos com alta performance.

No modelo treinado para separar **Melhores adequadas** e **Piores adequadas**, foi obtida acurácia de aproximadamente **97,88% no treino** e **97,21% no teste**, com valores equilibrados de precisão, recall e F1-score. Isso indica que, mesmo dentro da classe considerada adequada, existem diferenças internas consistentes que podem ser aprendidas pelo modelo.

De forma semelhante, no modelo treinado para separar **Melhores não adequadas** e **Piores não adequadas**, a acurácia foi de aproximadamente **98,59% no treino** e **96,82% no teste**, também com métricas bastante equilibradas. Esse resultado demonstra que a classe não adequada também não é homogênea, apresentando níveis distintos de criticidade.

A interpretação mais importante desses resultados é que os dados parecem seguir uma estrutura mais próxima de um espectro contínuo de qualidade da água, e não exatamente três blocos rígidos e perfeitamente separados. Essa estrutura pode ser representada da seguinte forma:

```text
Melhores adequadas
        ↓
Piores adequadas
        ↓
Melhores não adequadas
        ↓
Piores não adequadas
```

Nesse sentido, as classes **Piores adequadas** e **Melhores não adequadas** representam justamente a região intermediária do problema. Elas estão mais próximas da fronteira entre o que é considerado adequado e não adequado, funcionando como uma zona de transição. Já as **Melhores adequadas** representam os casos de melhor qualidade, enquanto as **Piores não adequadas** representam os casos mais críticos.

Essa abordagem ajuda a explicar por que o modelo de três classes apresentava dificuldade: a antiga classe **Boa** tentava representar diretamente essa região intermediária, mas essa separação não era suficientemente clara para o modelo. Ao trabalhar primeiro com uma classificação binária e, depois, analisar os subgrupos internos, foi possível capturar essa transição de forma mais organizada e interpretável.

Assim, para o objetivo do projeto, a estratégia mais adequada não é necessariamente utilizar um único modelo multiclasse para classificar diretamente a água como **Adequada**, **Boa** ou **Não adequada**. Uma alternativa mais robusta é utilizar uma abordagem hierárquica:

1. Primeiro, um modelo binário identifica se a amostra é **Adequada** ou **Não adequada**.
2. Em seguida, modelos auxiliares analisam o nível interno de qualidade dentro de cada classe.
3. A região intermediária pode ser interpretada a partir das amostras classificadas como **Piores adequadas** e **Melhores não adequadas**.

Com isso, o sistema consegue manter uma classificação principal mais confiável e, ao mesmo tempo, oferecer uma leitura mais detalhada sobre a condição ambiental da água. Essa abordagem é especialmente útil para o AquaSense, pois permite diferenciar casos realmente críticos daqueles que estão em uma região de alerta ou transição, apoiando uma tomada de decisão mais precisa no monitoramento ambiental.

Portanto, os experimentos indicam que a classificação binária apresentou melhor desempenho por reduzir a ambiguidade entre as classes, enquanto os modelos de subgrupos permitiram recuperar a noção de gradação da qualidade da água de maneira indireta. Essa combinação torna a solução mais coerente com a natureza dos dados e mais alinhada ao objetivo do projeto.
